[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/44.%20The%20Carbon%20Footprint%20Modeling%20Problem/P44-Tier-4_executed.ipynb)

---


# 44. The Carbon Footprint Modeling Problem

## Tier 4: Neural Architecture Search for Carbon Prediction

### Goal

This notebook uses **Neural Architecture Search (NAS)** to select a dense neural-network architecture for predicting supply-network carbon emissions. NAS is the process of evaluating candidate network structures and choosing one according to a specified validation metric. Here, candidate architectures are generated by random sampling, trained with numerical finite-difference gradients, and ranked by mean absolute error (MAE).

The task is **prediction**, not operational optimization: the model estimates emissions from input features, but it does not choose routes, transport modes, or facility decisions.

### Inputs and outputs

The input is a synthetic dataset containing ten supply-network features and a carbon-emissions target in kg CO₂e. The search evaluates a small, reproducible subset of this dataset. Its outputs are the best sampled architecture, validation metrics, computational measurements, and an illustrative prediction for a specified feature vector.

### Scope and modeling assumptions

- The dataset is synthetic. Its emissions are generated from a known formula with added random noise; it is not evidence of performance on measured supply-chain data.
- The search space in this experiment contains dense feed-forward networks. Recurrent and attention layers are not included because the network implementation below supports dense layers only.
- The input data are treated as independent rows. The experiment does not model time series, network topology, or dependencies between shipments.
- The final emissions layer is linear, as appropriate for a regression output that is not restricted to a fixed interval.
- A single held-out split is used for this compact demonstration. It is not cross-validation, and the held-out metric is used to select among candidates, so it is not an independent final test estimate.
- The validation MAE is an average absolute error, not a calibrated prediction interval or a statistical uncertainty estimate.

### Search and training method

For each trial, NAS samples a number of hidden layers, their widths, and their activation functions from the configured search space. The corresponding network is initialized, trained on the training split, and scored on the held-out split. The architecture with the lowest validation MAE is retained. This is random architecture sampling; no guided optimizer or alternative search method is used.

For a network with parameters \(\theta\), training minimizes mean squared error (MSE):

$$
L(\theta) = \frac{1}{n}\sum_{i=1}^{n}\left(f_{\theta}(\mathbf{x}_i)-y_i\right)^2.
$$

The notebook estimates each parameter's derivative by a forward finite difference,

$$
\frac{\partial L}{\partial \theta_j} \approx
\frac{L(\theta + \varepsilon\mathbf{e}_j)-L(\theta)}{\varepsilon},
$$

and updates it by gradient descent, \(\theta_j \leftarrow \theta_j-\eta\,\partial L/\partial\theta_j\). Here, \(n\) is the number of training rows, \(f_{\theta}\) is the network prediction, \(\varepsilon\) is a small perturbation, \(\mathbf{e}_j\) selects parameter \(j\), and \(\eta\) is the learning rate. The implementation standardizes the target using training data, which keeps its scale manageable during numerical optimization; reported MAE and \(R^2\) are calculated in the original target units.

A candidate with \(H\) hidden layers and widths \(h_1,\ldots,h_H\), input dimension \(d\), and one output has \(d h_1 + \sum_{k=1}^{H-1}h_kh_{k+1}+h_H + h_1 + \cdots + h_H + 1\) trainable parameters (weights and biases). Finite-difference training requires a loss evaluation for each parameter in each epoch, so its computational cost grows with both the training-set size and the number of parameters. This implementation is intended for a small demonstration, not large-scale model training.

### How to interpret the results

Lower validation MAE means smaller average absolute prediction errors on the held-out rows. An \(R^2\) value near 1 indicates that predictions explain much of the target variation on that split; a negative value means the predictions perform worse than the constant-mean reference for that split. Trial-to-trial differences can reflect both architecture and random initialization. The results should therefore be read as an outcome of this specific synthetic-data experiment, not as a general performance guarantee.

In [ ]:
# Import the small set of tools used for data generation, model evaluation, and plotting.
import random
import time
from dataclasses import dataclass
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

plt.style.use("default")
sns.set_palette("husl")

## 1. Define a simple neural network

A feed-forward network repeatedly applies an affine transformation and, in hidden layers, a nonlinear activation. For layer \(\ell\), the calculation is \(\mathbf{a}^{(\ell)}=g_{\ell}(\mathbf{a}^{(\ell-1)}W^{(\ell)}+\mathbf{b}^{(\ell)})\). The input vector is \(\mathbf{a}^{(0)}\); \(W^{(\ell)}\) and \(\mathbf{b}^{(\ell)}\) are the layer's weights and biases; and \(g_{\ell}\) is its activation function. The output layer is linear so the model can predict a continuous emissions value.

This implementation makes the model-building step explicit: the architecture dictionary determines each layer's width and activation, while `forward` evaluates the resulting network. Training uses finite-difference estimates of the MSE gradient, as described above.

In [ ]:
class SimpleNeuralNetwork:
    """Dense feed-forward network trained with finite-difference gradients."""

    def __init__(self, architecture: Dict):
        self.architecture = architecture
        self.weights = []
        self.biases = []
        self.activations = []
        self._build_network()

    def _build_network(self) -> None:
        """Initialize each affine layer using Xavier uniform initialization."""
        layer_sizes = self.architecture["layer_sizes"]
        activation_funcs = self.architecture["activations"]

        if len(layer_sizes) < 2:
            raise ValueError("An architecture must contain input and output layers.")
        if len(activation_funcs) != len(layer_sizes) - 1:
            raise ValueError("There must be one activation per affine layer.")

        for input_size, output_size, activation in zip(
            layer_sizes[:-1], layer_sizes[1:], activation_funcs
        ):
            limit = np.sqrt(6.0 / (input_size + output_size))
            self.weights.append(
                np.random.uniform(-limit, limit, (input_size, output_size))
            )
            self.biases.append(np.zeros(output_size))
            self.activations.append(activation)

    @staticmethod
    def _activation(values: np.ndarray, function: str) -> np.ndarray:
        """Apply a supported hidden-layer activation function."""
        if function == "relu":
            return np.maximum(0.0, values)
        if function == "sigmoid":
            values = np.clip(values, -500.0, 500.0)
            return 1.0 / (1.0 + np.exp(-values))
        if function == "tanh":
            return np.tanh(values)
        if function == "swish":
            clipped = np.clip(values, -500.0, 500.0)
            return values / (1.0 + np.exp(-clipped))
        if function == "gelu":
            return 0.5 * values * (
                1.0
                + np.tanh(
                    np.sqrt(2.0 / np.pi)
                    * (values + 0.044715 * values**3)
                )
            )
        if function == "linear":
            return values
        raise ValueError(f"Unsupported activation function: {function}")

    def forward(self, X: np.ndarray) -> np.ndarray:
        """Return predictions for a two-dimensional feature array."""
        current = np.asarray(X, dtype=float)
        if current.ndim != 2:
            raise ValueError("X must be a two-dimensional array.")

        for weights, bias, activation in zip(
            self.weights, self.biases, self.activations
        ):
            current = current @ weights + bias
            current = self._activation(current, activation)
        return current

    def train(
        self,
        X: np.ndarray,
        y: np.ndarray,
        epochs: int = 100,
        learning_rate: float = 0.01,
        verbose: bool = False,
    ) -> List[float]:
        """Train by coordinate-wise forward finite differences and gradient descent."""
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).reshape(-1, 1)
        if X.shape[0] != y.shape[0]:
            raise ValueError("X and y must contain the same number of rows.")

        losses = []
        epsilon = 1e-5

        for epoch in range(epochs):
            epoch_loss = float(np.mean((self.forward(X) - y) ** 2))

            # Each perturbation estimates one coordinate of the current loss gradient.
            for layer_idx in range(len(self.weights)):
                weight_grad = np.zeros_like(self.weights[layer_idx])
                bias_grad = np.zeros_like(self.biases[layer_idx])

                for row in range(self.weights[layer_idx].shape[0]):
                    for column in range(self.weights[layer_idx].shape[1]):
                        original = self.weights[layer_idx][row, column]
                        current_loss = np.mean((self.forward(X) - y) ** 2)
                        self.weights[layer_idx][row, column] = original + epsilon
                        perturbed_loss = np.mean((self.forward(X) - y) ** 2)
                        self.weights[layer_idx][row, column] = original
                        weight_grad[row, column] = (
                            perturbed_loss - current_loss
                        ) / epsilon

                for column in range(self.biases[layer_idx].shape[0]):
                    original = self.biases[layer_idx][column]
                    current_loss = np.mean((self.forward(X) - y) ** 2)
                    self.biases[layer_idx][column] = original + epsilon
                    perturbed_loss = np.mean((self.forward(X) - y) ** 2)
                    self.biases[layer_idx][column] = original
                    bias_grad[column] = (perturbed_loss - current_loss) / epsilon

                self.weights[layer_idx] -= learning_rate * weight_grad
                self.biases[layer_idx] -= learning_rate * bias_grad

            losses.append(epoch_loss)
            if verbose and epoch % 20 == 0:
                print(f"Epoch {epoch}: MSE = {epoch_loss:.6f}")

        return losses

## 2. Specify the search configuration and result records

The configuration describes which architectures can be sampled. In this experiment, `min_layers` and `max_layers` count hidden layers; the input and output layers are added separately. Each hidden layer receives a width from `layer_sizes` and an activation from `activations`. The output layer is always a single linear unit for regression.

The result record stores the sampled architecture and its validation MAE, validation \(R^2\), per-sample inference time, training time, and parameter count. Keeping these measurements together makes it possible to rank candidates and inspect the search without confusing model quality with computational cost.

In [ ]:
@dataclass
class NASConfig:
    """Choices that define the dense-network architecture search space."""

    max_layers: int = 6
    min_layers: int = 2
    layer_types: List[str] = None
    activations: List[str] = None
    layer_sizes: List[int] = None

    def __post_init__(self):
        if self.layer_types is None:
            self.layer_types = ["dense"]
        if self.activations is None:
            self.activations = ["relu", "swish", "gelu", "tanh"]
        if self.layer_sizes is None:
            self.layer_sizes = [32, 64, 128, 256]
        if self.min_layers < 1 or self.max_layers < self.min_layers:
            raise ValueError("Layer bounds must satisfy 1 <= min_layers <= max_layers.")
        if set(self.layer_types) != {"dense"}:
            raise ValueError("This network implementation supports dense layers only.")
        if not self.activations or not self.layer_sizes:
            raise ValueError("The activation and layer-size choices cannot be empty.")


@dataclass
class ArchitectureResult:
    """Validation and computational measurements for one sampled architecture."""

    architecture: Dict
    mae: float
    r2: float
    inference_time: float
    training_time: float
    num_parameters: int


@dataclass
class CarbonDataset:
    """Feature matrix, emissions targets, and descriptions for the dataset."""

    features: np.ndarray
    targets: np.ndarray
    feature_names: List[str]
    description: str

## 3. Create the synthetic carbon dataset

The dataset generator creates ten features, including distance, transport-mode shares, facility count, utilization, weather, traffic, time of day, and season. Emissions are generated from a transparent formula: distance and transport mix contribute a base amount; facilities contribute an additional amount; weather, traffic, time, and season modify the total; and Gaussian noise represents unmodeled variation.

This construction is useful for demonstrating a complete prediction workflow because both features and targets are available in a reproducible form. Since the data are synthetic and generated from a known formula, performance on them should not be interpreted as validation on real operations.

In [ ]:
def create_synthetic_carbon_dataset(
    n_samples: int = 50000,
) -> CarbonDataset:
    """Generate synthetic supply-network features and carbon-emissions targets."""
    np.random.seed(42)

    total_distance = np.random.uniform(100, 5000, n_samples)
    truck_percentage = np.random.uniform(0, 1, n_samples)
    rail_percentage = np.random.uniform(0, 1 - truck_percentage, n_samples)
    ship_percentage = 1 - truck_percentage - rail_percentage

    active_facilities = np.random.randint(1, 10, n_samples)
    capacity_utilization = np.random.uniform(0.3, 0.95, n_samples)
    weather_conditions = np.random.uniform(0, 1, n_samples)
    traffic_density = np.random.uniform(0.1, 1.0, n_samples)
    time_of_day = np.random.uniform(0, 24, n_samples)
    season = np.random.randint(0, 4, n_samples)

    features = np.column_stack(
        [
            total_distance,
            truck_percentage,
            rail_percentage,
            ship_percentage,
            active_facilities,
            capacity_utilization,
            weather_conditions,
            traffic_density,
            time_of_day,
            season,
        ]
    )

    base_emissions = total_distance * (
        0.8 * truck_percentage
        + 0.3 * rail_percentage
        + 0.15 * ship_percentage
    )
    facility_emissions = active_facilities * (
        50 + 30 * capacity_utilization
    )
    weather_factor = 1 + 0.2 * weather_conditions
    traffic_factor = 1 + 0.15 * traffic_density
    time_factor = 1 + 0.1 * np.sin(2 * np.pi * time_of_day / 24)
    season_factor = 1 + 0.05 * (season - 1.5)

    targets = (
        (base_emissions + facility_emissions)
        * weather_factor
        * traffic_factor
        * time_factor
        * season_factor
    )
    targets += np.random.normal(0, 50, n_samples)
    targets = np.maximum(targets, 100)

    feature_names = [
        "total_distance",
        "truck_percentage",
        "rail_percentage",
        "ship_percentage",
        "active_facilities",
        "capacity_utilization",
        "weather_conditions",
        "traffic_density",
        "time_of_day",
        "season",
    ]

    return CarbonDataset(
        features=features,
        targets=targets,
        feature_names=feature_names,
        description=f"Synthetic carbon footprint dataset with {n_samples} samples",
    )


dataset = create_synthetic_carbon_dataset(50000)
print(f"Dataset: {dataset.description}")
print(f"Feature matrix shape: {dataset.features.shape}")
print(f"Target vector shape: {dataset.targets.shape}")
print(f"Feature names: {dataset.feature_names}")
print(
    "Target range: "
    f"{dataset.targets.min():.1f} to {dataset.targets.max():.1f} kg CO₂e"
)

## 4. Generate and describe candidate architectures

A sampled architecture is represented by its layer widths, layer types, and activations. For example, a width list \([d,h_1,h_2,1]\) means an input layer of size \(d\), two hidden layers of widths \(h_1\) and \(h_2\), and one output. The architecture generator samples the hidden structure while fixing the final output width and activation for regression.

The parameter-count function sums the number of weights and biases in every dense connection. It is a structural measure of model size, not a direct measurement of energy use or inference latency.

In [ ]:
def generate_random_architecture(
    config: NASConfig, input_dim: int
) -> Dict:
    """Sample a dense-network architecture from the configured search space."""
    hidden_layer_count = random.randint(config.min_layers, config.max_layers)
    layer_sizes = [input_dim]
    layer_types = []
    activations = []

    for _ in range(hidden_layer_count):
        layer_sizes.append(random.choice(config.layer_sizes))
        layer_types.append("dense")
        activations.append(random.choice(config.activations))

    layer_sizes.append(1)
    layer_types.append("dense")
    activations.append("linear")

    return {
        "layer_sizes": layer_sizes,
        "layer_types": layer_types,
        "activations": activations,
        "num_layers": hidden_layer_count,
    }


def calculate_num_parameters(architecture: Dict) -> int:
    """Count dense-layer weights and biases in an architecture."""
    layer_sizes = architecture["layer_sizes"]
    return sum(
        input_size * output_size + output_size
        for input_size, output_size in zip(layer_sizes[:-1], layer_sizes[1:])
    )


print("Architecture sampling and parameter-count functions are ready.")

## 5. Train and evaluate one candidate

Each candidate is initialized and trained using the training split. The validation predictions are converted back to kg CO₂e before MAE and \(R^2\) are calculated, so the reported errors have an interpretable physical unit. Inference time is measured for the full validation batch and divided by its number of rows to obtain milliseconds per sample.

The training target is standardized using its training-set mean and standard deviation. This scaling changes the numerical units used internally for optimization, not the underlying regression objective. The returned evaluation record contains the original-scale validation metrics.

In [ ]:
def evaluate_architecture(
    architecture: Dict,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
    max_epochs: int = 50,
) -> ArchitectureResult:
    """Train one candidate and measure its validation performance and cost."""
    model = SimpleNeuralNetwork(architecture)

    target_mean = float(np.mean(y_train))
    target_scale = float(np.std(y_train))
    if target_scale == 0:
        target_scale = 1.0
    y_train_scaled = (y_train - target_mean) / target_scale

    training_start = time.perf_counter()
    model.train(
        X_train,
        y_train_scaled,
        epochs=max_epochs,
        learning_rate=0.01,
        verbose=False,
    )
    training_time = time.perf_counter() - training_start

    inference_start = time.perf_counter()
    predictions_scaled = model.forward(X_val).reshape(-1)
    elapsed = time.perf_counter() - inference_start
    predictions = predictions_scaled * target_scale + target_mean
    inference_time = elapsed / len(X_val) * 1000.0

    return ArchitectureResult(
        architecture=architecture,
        mae=float(mean_absolute_error(y_val, predictions)),
        r2=float(r2_score(y_val, predictions)),
        inference_time=float(inference_time),
        training_time=float(training_time),
        num_parameters=calculate_num_parameters(architecture),
    )


print("Candidate evaluation function is ready.")

## 6. Run random architecture search

The search evaluates each randomly sampled candidate on the same training and validation rows. It retains the candidate with the smallest validation MAE. If multiple candidates have equal MAE to floating-point precision, the first encountered candidate remains the selected result; ties are not evidence that the architectures are identical.

Only a small subset of the generated dataset is used here to keep numerical finite-difference training manageable. The random seeds make the dataset split, architecture sampling, and weight initialization reproducible within the same software environment. The `cv_folds` argument is retained as a descriptive setting for the search report; this implementation uses one held-out split, not cross-validation.

In [ ]:
def neural_architecture_search(
    dataset: CarbonDataset,
    config: NASConfig,
    n_trials: int = 5,
    cv_folds: int = 1,
) -> Tuple[ArchitectureResult, List[ArchitectureResult]]:
    """Sample, train, and rank candidate architectures by held-out MAE."""
    if n_trials < 1:
        raise ValueError("n_trials must be at least 1.")
    if cv_folds != 1:
        raise ValueError("This experiment uses one held-out split, not cross-validation.")

    print("=" * 80)
    print("NEURAL ARCHITECTURE SEARCH FOR CARBON FOOTPRINT PREDICTION")
    print("=" * 80)
    print(f"Dataset: {dataset.description}")
    print(
        "Hidden layers: "
        f"{config.min_layers} to {config.max_layers}"
    )
    print(f"Layer types: {config.layer_types}")
    print(f"Activations: {config.activations}")
    print(f"Trials: {n_trials}")
    print("Evaluation: one held-out validation split")

    X = dataset.features[:200]
    y = dataset.targets[:200]
    X_train_raw, X_val_raw, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # Fit preprocessing only on training rows to avoid leaking validation statistics.
    feature_scaler = StandardScaler()
    X_train = feature_scaler.fit_transform(X_train_raw)
    X_val = feature_scaler.transform(X_val_raw)

    random.seed(42)
    np.random.seed(42)
    all_results = []

    print("\nSampling and evaluating architectures...")
    for trial in range(n_trials):
        architecture = generate_random_architecture(config, X_train.shape[1])
        result = evaluate_architecture(
            architecture,
            X_train,
            y_train,
            X_val,
            y_val,
        )
        all_results.append(result)
        print(
            f"Trial {trial + 1:2d}: MAE = {result.mae:.1f} kg CO₂e, "
            f"R² = {result.r2:.3f}"
        )

    best_result = min(all_results, key=lambda result: result.mae)
    print(f"\nSearch complete. Best validation MAE: {best_result.mae:.1f} kg CO₂e")
    return best_result, all_results


nas_config = NASConfig(
    max_layers=2,
    min_layers=2,
    layer_types=["dense"],
    activations=["relu"],
    layer_sizes=[4, 8],
)

best_result, all_results = neural_architecture_search(
    dataset,
    nas_config,
    n_trials=5,
    cv_folds=1,
)

## 7. Inspect the selected architecture and search results

This report summarizes the architecture selected by validation MAE and the distribution of results across the sampled candidates. MAE is measured in kg CO₂e; \(R^2\) is unitless; inference time is an empirical timing measurement and can vary with hardware and runtime conditions. Search statistics describe only the candidates actually evaluated in this run.

In [ ]:
def analyze_nas_results(
    best_result: ArchitectureResult,
    all_results: List[ArchitectureResult],
) -> None:
    """Report the selected architecture and descriptive search statistics."""
    if not all_results:
        raise ValueError("At least one architecture result is required.")

    architecture = best_result.architecture
    maes = [result.mae for result in all_results]
    r2_values = [result.r2 for result in all_results]
    parameter_counts = [result.num_parameters for result in all_results]

    print("\n" + "=" * 80)
    print("NEURAL ARCHITECTURE SEARCH RESULTS")
    print("=" * 80)
    print("\nSelected architecture")
    print(f"Hidden layers: {architecture['num_layers']}")
    print(f"Layer sizes, including input and output: {architecture['layer_sizes']}")
    print(f"Layer types: {architecture['layer_types']}")
    print(f"Activations: {architecture['activations']}")

    print("\nValidation and computational measurements")
    print(f"MAE: {best_result.mae:.1f} kg CO₂e")
    print(f"R²: {best_result.r2:.3f}")
    print(f"Inference time: {best_result.inference_time:.4f} ms per sample")
    print(f"Training time: {best_result.training_time:.3f} seconds")
    print(f"Trainable parameters: {best_result.num_parameters:,}")

    print("\nSearch summary")
    print(f"Architectures evaluated: {len(all_results)}")
    print(f"Validation MAE range: {min(maes):.1f} to {max(maes):.1f} kg CO₂e")
    print(f"Validation R² range: {min(r2_values):.3f} to {max(r2_values):.3f}")
    print(
        "Parameter-count range: "
        f"{min(parameter_counts):,} to {max(parameter_counts):,}"
    )
    print(f"Validation MAE standard deviation: {np.std(maes):.1f} kg CO₂e")


analyze_nas_results(best_result, all_results)

## 8. Make an illustrative prediction

The selected architecture is initialized and trained on the same training partition used in the search. A user-specified feature vector is transformed with a feature scaler fitted only on those training rows. The predicted target is transformed back to kg CO₂e.

For context, the function also reports the selected candidate's validation MAE. That value summarizes average validation error; it is not a guarantee that this individual prediction is within that distance of its unknown true value. The example is generated from the synthetic dataset's feature definitions and should not be read as a measured real-world shipment.

In [ ]:
def real_world_prediction_example(
    best_result: ArchitectureResult,
    dataset: CarbonDataset,
) -> Tuple[SimpleNeuralNetwork, StandardScaler]:
    """Train the selected architecture and predict for an illustrative feature row."""
    X = dataset.features[:200]
    y = dataset.targets[:200]
    X_train_raw, _, y_train, _ = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    feature_scaler = StandardScaler()
    X_train = feature_scaler.fit_transform(X_train_raw)
    target_mean = float(np.mean(y_train))
    target_scale = float(np.std(y_train))
    if target_scale == 0:
        target_scale = 1.0
    y_train_scaled = (y_train - target_mean) / target_scale

    model = SimpleNeuralNetwork(best_result.architecture)
    model.train(
        X_train,
        y_train_scaled,
        epochs=50,
        learning_rate=0.01,
        verbose=False,
    )

    example_features = {
        "total_distance": 2450,
        "truck_percentage": 0.40,
        "rail_percentage": 0.60,
        "ship_percentage": 0.00,
        "active_facilities": 3,
        "capacity_utilization": 0.78,
        "weather_conditions": 0.3,
        "traffic_density": 0.5,
        "time_of_day": 14.0,
        "season": 2,
    }
    example_array = np.array(
        [[example_features[name] for name in dataset.feature_names]],
        dtype=float,
    )
    example_scaled = feature_scaler.transform(example_array)

    prediction_start = time.perf_counter()
    prediction_scaled = float(model.forward(example_scaled)[0, 0])
    prediction = prediction_scaled * target_scale + target_mean
    inference_time = (time.perf_counter() - prediction_start) * 1000.0

    print("\n" + "=" * 80)
    print("ILLUSTRATIVE SYNTHETIC-DATA PREDICTION")
    print("=" * 80)
    print("Input features")
    for feature, value in example_features.items():
        if feature in {"truck_percentage", "rail_percentage", "ship_percentage"}:
            print(f"{feature}: {value:.0%}")
        elif feature in {"capacity_utilization", "weather_conditions", "traffic_density"}:
            print(f"{feature}: {value:.0%}")
        else:
            print(f"{feature}: {value}")

    print("\nPrediction")
    print(f"Estimated emissions: {prediction:.1f} kg CO₂e")
    print(f"Selected architecture validation MAE: {best_result.mae:.1f} kg CO₂e")
    print(f"Single-row inference time: {inference_time:.4f} ms")
    print(
        "The validation MAE is an average error measure, not a prediction interval."
    )
    return model, feature_scaler


best_model, scaler = real_world_prediction_example(best_result, dataset)

## 9. Visualize search outcomes

The plots show validation MAE over the trial sequence, the relationship between validation MAE and \(R^2\), and the sampled architectures' parameter counts. The architecture diagram labels each dense layer by width and activation. In the diagram, arrows indicate the forward direction from input to output; the output layer uses a linear activation.

In [ ]:
def visualize_nas_results(
    best_result: ArchitectureResult,
    all_results: List[ArchitectureResult],
) -> None:
    """Plot search metrics, model size, and the selected network structure."""
    if not all_results:
        raise ValueError("At least one architecture result is required.")

    fig, axes = plt.subplots(2, 2, figsize=(13, 10))
    fig.suptitle(
        "Neural Architecture Search for Carbon Footprint Prediction",
        fontsize=15,
        fontweight="bold",
    )

    maes = [result.mae for result in all_results]
    r2_values = [result.r2 for result in all_results]
    trials = np.arange(1, len(all_results) + 1)

    ax = axes[0, 0]
    running_best = np.minimum.accumulate(maes)
    ax.scatter(trials, maes, color="steelblue", label="Trial validation MAE")
    ax.plot(trials, running_best, color="darkorange", label="Best MAE so far")
    ax.set_xlabel("Trial")
    ax.set_ylabel("Validation MAE (kg CO₂e)")
    ax.set_title("Search progress")
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[0, 1]
    ax.scatter(maes, r2_values, color="seagreen", edgecolors="black")
    ax.scatter(
        best_result.mae,
        best_result.r2,
        color="crimson",
        marker="*",
        s=180,
        label="Selected architecture",
        zorder=3,
    )
    ax.set_xlabel("Validation MAE (kg CO₂e)")
    ax.set_ylabel("Validation R²")
    ax.set_title("Validation metrics by candidate")
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[1, 0]
    parameter_counts = [result.num_parameters for result in all_results]
    ax.bar(trials, parameter_counts, color="mediumpurple", edgecolor="black")
    ax.set_xlabel("Trial")
    ax.set_ylabel("Trainable parameters")
    ax.set_title("Candidate model size")
    ax.grid(True, alpha=0.3, axis="y")

    ax = axes[1, 1]
    architecture = best_result.architecture
    layer_sizes = architecture["layer_sizes"]
    activations = architecture["activations"]
    layer_positions = np.arange(len(layer_sizes))
    ax.plot(layer_positions, np.zeros_like(layer_positions), color="gray", zorder=1)

    for index, (size, activation) in enumerate(zip(layer_sizes, activations)):
        label = f"Dense({size})\n{activation}"
        ax.scatter(index, 0, s=900, color="lightblue", edgecolors="black", zorder=2)
        ax.annotate(label, (index, 0), ha="center", va="center", fontsize=8, zorder=3)

    ax.set_xlim(-0.6, len(layer_sizes) - 0.4)
    ax.set_ylim(-0.7, 0.7)
    ax.set_xticks(layer_positions)
    ax.set_xticklabels(
        ["Input"]
        + [f"Hidden {index}" for index in range(1, len(layer_sizes) - 1)]
        + ["Output"]
    )
    ax.set_yticks([])
    ax.set_title("Selected architecture: forward layer order")
    ax.set_xlabel("Layer")

    plt.tight_layout()
    plt.show()


visualize_nas_results(best_result, all_results)

## 10. Computational interpretation and limitations

NAS spends computation evaluating candidate architectures. In this implementation, each trial entails training one network and scoring it on the same held-out split. Training time is measured directly for each candidate; it does not include every possible notebook or plotting overhead. The number of model evaluations is therefore the number of architecture trials, not a cross-validation count.

Several limitations matter when interpreting the outcome:

- Random sampling explores only the architectures drawn in the finite set of trials; it does not establish a global optimum.
- The selected validation score is used during architecture selection and can therefore be optimistic as an estimate of performance on new data.
- Finite-difference gradient estimation is computationally expensive as the number of parameters grows.
- This experiment uses dense layers and a small data subset. It does not evaluate recurrent or attention-based architectures, despite those being possible topics in broader NAS research.
- The synthetic data-generating process is simpler and more controlled than real supply-chain measurement and reporting.
- Validation MAE is not uncertainty quantification, and this notebook does not produce a calibrated interval for an individual prediction.

A useful reading of the output is therefore: among the sampled dense architectures, which candidate had the lowest MAE on this particular held-out synthetic-data split, and what were its associated model size and measured runtime?

In [ ]:
def nas_performance_analysis(
    best_result: ArchitectureResult,
    all_results: List[ArchitectureResult],
) -> None:
    """Summarize observed search measurements and the method's scope."""
    if not all_results:
        raise ValueError("At least one architecture result is required.")

    total_training_time = sum(result.training_time for result in all_results)
    average_training_time = total_training_time / len(all_results)

    print("=" * 80)
    print("NEURAL ARCHITECTURE SEARCH: COMPUTATIONAL SUMMARY")
    print("=" * 80)
    print(f"Architecture trials: {len(all_results)}")
    print(f"Model evaluations: {len(all_results)}")
    print(f"Total candidate training time: {total_training_time:.3f} seconds")
    print(f"Average candidate training time: {average_training_time:.3f} seconds")
    print(f"Selected validation MAE: {best_result.mae:.1f} kg CO₂e")
    print(f"Selected validation R²: {best_result.r2:.3f}")
    print(f"Selected model parameters: {best_result.num_parameters:,}")
    print("Evaluation design: one held-out split; no cross-validation")
    print("Data scope: synthetic carbon-emissions dataset")
    print("Search strategy: random architecture sampling")
    print("Training method: finite-difference gradients with gradient descent")
    print(
        "Interpretation: these measurements describe this run and do not establish "
        "real-world predictive performance."
    )


nas_performance_analysis(best_result, all_results)

## Summary

This notebook demonstrates a complete, small-scale NAS workflow for carbon-footprint prediction: define a dense-network search space, generate candidate architectures, train each candidate, rank them by held-out MAE, inspect the selected architecture, and make an illustrative prediction. The optimization objective used for training is MSE, while candidate selection uses validation MAE; these metrics serve different roles and are both reported in their appropriate units.

The method is educational rather than production-ready. The finite-difference training procedure and small random search are computationally limited, the validation split is reused for candidate selection, and the data are synthetic. Accordingly, the selected architecture is the best among the candidates sampled in this experiment—not a proven global optimum or a validated model for measured supply-chain emissions.